### The objective of this notebook is to implement LightGBM global forecasting model, and compare the results with those of statistical models. Unlike the statistical models, LightGBM will be trained across the entire 228 series simultaneously, like a tabular dataset along with some engineered features. 

In [2]:
# Importing the libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import lightgbm as lgb
from sklearn.preprocessing import LabelEncoder

from hierarchicalforecast.core import HierarchicalReconciliation
from hierarchicalforecast.methods import MinTrace

In [3]:
# Loading the hierarchy ready dataset

df = pd.read_csv('../data/processed/hierarchy_ready.csv')
df['ds'] = pd.to_datetime(df['ds'])

# Dropping the 4 series
to_drop = {
    'TOTAL/MUMBAI/MUMBAI-RAJKOT',
    'TOTAL/DELHI/DELHI-RAJKOT',
    'TOTAL/DEHRA DUN/DEHRA DUN-DELHI',
    'TOTAL/DEHRA DUN'
}

df = df[~df['unique_id'].isin(to_drop)].copy()

# Sorting the values for lag construction
df = df.sort_values(['unique_id','ds']).reset_index(drop = True)

print('Shape: ',df.shape)
print('Series: ',df['unique_id'].nunique())
print('Date Range: ',df['ds'].min().date(),' to ',df['ds'].max().date())
print(df.head())

Shape:  (26677, 3)
Series:  228
Date Range:  2015-04-01  to  2026-02-01
  unique_id         ds          y
0     TOTAL 2015-04-01  2364483.0
1     TOTAL 2015-05-01  2547864.0
2     TOTAL 2015-06-01  2327666.0
3     TOTAL 2015-07-01  2431149.0
4     TOTAL 2015-08-01  2381990.0


In [4]:
# Feature Engineering 01 - Lag
for lag in [1,2,3,12]:
    df[f'y_lag_{lag}'] = df.groupby('unique_id')['y'].shift(lag)

# check for NaN values in lag columns
print("NaN count after lag feature construction: ")
print(df[['y_lag_1','y_lag_2','y_lag_3','y_lag_12']].isna().sum())

NaN count after lag feature construction: 
y_lag_1      228
y_lag_2      456
y_lag_3      684
y_lag_12    2736
dtype: int64


In [5]:
# Rolling means per series
df['y_rollmean_3'] = (df.groupby('unique_id')['y'].transform(lambda s: s.shift(1).rolling(window = 3, min_periods = 1).mean()))
df['y_rollmean_12'] = (df.groupby('unique_id')['y'].transform(lambda s: s.shift(1).rolling(window = 12, min_periods = 1).mean()))

# Sanity checks
print("NaN counts after rolling means: ")
print(df[['y_rollmean_3','y_rollmean_12']].isna().sum())

# Re-verify on MUMBAI
sample = df[df['unique_id'] == 'TOTAL/MUMBAI'].head(5)[['ds','y','y_rollmean_3']]
print('MUMBAI airport first 5 rows: ')
print(sample.to_string(index = False))

NaN counts after rolling means: 
y_rollmean_3     228
y_rollmean_12    228
dtype: int64
MUMBAI airport first 5 rows: 
        ds        y  y_rollmean_3
2015-04-01 672098.0           NaN
2015-05-01 712385.0 672098.000000
2015-06-01 599957.0 692241.500000
2015-07-01 656874.0 661480.000000
2015-08-01 640882.0 656405.333333


In [6]:
# Calendar Features
df['month'] = df['ds'].dt.month
df['quarter'] = df['ds'].dt.quarter
df['year'] = df['ds'].dt.year

print(df[['ds','month','quarter','year']].head())

          ds  month  quarter  year
0 2015-04-01      4        2  2015
1 2015-05-01      5        2  2015
2 2015-06-01      6        2  2015
3 2015-07-01      7        3  2015
4 2015-08-01      8        3  2015


In [7]:
# Hierarchy level encoding
df['level'] = df['unique_id'].apply(
    lambda x: 0 if '/' not in x
    else (1 if x.count('/') == 1 else 2)
)

# Label encoding for unique id
le = LabelEncoder()
df['series_id'] = le.fit_transform(df['unique_id'])

print("Rows per level: ")
print(df['level'].value_counts().sort_index())
print(f"Unique series_id values: {df['series_id'].nunique()}")
print(df[['unique_id','level','series_id']].head())

Rows per level: 
level
0      131
1     3591
2    22955
Name: count, dtype: int64
Unique series_id values: 228
  unique_id  level  series_id
0     TOTAL      0          0
1     TOTAL      0          0
2     TOTAL      0          0
3     TOTAL      0          0
4     TOTAL      0          0


In [8]:
# Dropping NaN features
df_ml = df.dropna().reset_index(drop = True)
print(f"Rows before dropping NaN values: {len(df)}")
print(f"Rows after dropping NaN values: {len(df_ml)}")

Rows before dropping NaN values: 26677
Rows after dropping NaN values: 23941


In [9]:
# Test and Train Split
split_date = pd.Timestamp('2025-03-01')

train_ml = df_ml[df_ml['ds'] < split_date].copy()
test_ml = df_ml[df_ml['ds'] >= split_date].copy()

print(f"Train: {train_ml.shape} | Range: {train_ml['ds'].min().date()} to {train_ml['ds'].max().date()}")
print(f"Test: {test_ml.shape} | Range: {test_ml['ds'].min().date()} to {test_ml['ds'].max().date()}")

print(f"Series in Train dataset: {train_ml['unique_id'].nunique()}")
print(f"Series in Test dataset: {test_ml['unique_id'].nunique()}")

Train: (21205, 14) | Range: 2016-04-01 to 2025-02-01
Test: (2736, 14) | Range: 2025-03-01 to 2026-02-01
Series in Train dataset: 228
Series in Test dataset: 228


In [10]:
# Final checks for data sanilty
features = ['y_lag_1','y_lag_2','y_lag_3','y_lag_12',
            'y_rollmean_3','y_rollmean_12',
            'month','quarter','year',
            'level','series_id']

# Check for NaN Values
print(train_ml[features].isna().sum().sum(), " NaN values in training dataset")
print(test_ml[features].isna().sum().sum(), " NaN values in test dataset")

# Stats per feature
print(train_ml[features].describe().round(2).to_string())

# Traget column
print(f"Target y: {train_ml['ds'].min().date()} to {train_ml['ds'].max().date()}")
print(f"Test y: {test_ml['ds'].min().date()} to {test_ml['ds'].max().date()}")


0  NaN values in training dataset
0  NaN values in test dataset
          y_lag_1     y_lag_2     y_lag_3    y_lag_12  y_rollmean_3  y_rollmean_12     month   quarter      year     level  series_id
count    21205.00    21205.00    21205.00    21205.00      21205.00       21205.00  21205.00  21205.00  21205.00  21205.00   21205.00
mean     61474.79    61083.44    60701.76    57594.39      61086.67       59526.41      6.54      2.52   2020.58      1.85     114.68
std     318588.08   316367.54   313918.60   295010.77     314767.64      300294.48      3.46      1.12      2.53      0.37      66.37
min          0.00        0.00        0.00        0.00          0.00         141.92      1.00      1.00   2016.00      0.00       0.00
25%       6746.00     6653.50     6577.00     5958.00       6756.33        6870.00      4.00      2.00   2019.00      2.00      57.00
50%      14641.00    14537.00    14446.00    13683.00      14600.33       14424.92      7.00      3.00   2021.00      2.00     115.0

In [11]:
# Saving the files
train_ml.to_csv('../outputs/lightgbm_train.csv',index = False)
test_ml.to_csv('../outputs/lightgbm_test.csv',index = False)

In [13]:
X_train = train_ml[features]
y_train = train_ml['y']

model = lgb.LGBMRegressor(
    n_estimators = 500,
    learning_rate = 0.05,
    num_leaves = 63,
    min_child_samples = 20,
    random_state = 42,
    verbose = -1
)

model.fit(X_train, y_train)

print("LightGBM model trained on ", len(X_train), " rows", " across ", len(features), " features")

LightGBM model trained on  21205  rows  across  11  features


In [16]:
# Recursive 12-month forecast
full = pd.concat([train_ml, test_ml], ignore_index = True).sort_values(['unique_id','ds']).reset_index(drop = True)

test_dates = sorted(test_ml['ds'].unique())

preds = {}

for d in test_dates:
    mask = full['ds'] == d
    X_pred = full.loc[mask, features]
    y_hat = model.predict(X_pred)

    # Store predictions
    for uid, val in zip(full.loc[mask, 'unique_id'], y_hat):
        preds [(uid, d)] = val
    
    # Update the y value for these rows so that downstraem lags can be computed using these predictions
    full.loc[mask, 'y'] = y_hat

    # Recompute lag features for the next month using the updated y
    next_idx = test_dates.index(d) + 1
    if next_idx < len(test_dates):
        next_d = test_dates[next_idx]
        next_mask = full['ds'] == next_d

        # For each series, find the y values at lag 1, 2, 3, 12 from next_d
        for uid in full.loc[next_mask, 'unique_id'].unique():
            series = full[full['unique_id'] == uid].sort_values('ds')
            row_idx = series[series['ds'] == next_d].index[0]
            for lag in [1, 2, 3, 12]:
                lag_val = series[series['ds'] == next_d - pd.DateOffset(months=lag)]['y']
                if len(lag_val) > 0:
                    full.loc[row_idx, f'y_lag_{lag}'] = lag_val.values[0]
            
            # Rolling means
            past = series[series['ds'] < next_d]['y']
            full.loc[row_idx, 'y_rollmean_3'] = past.tail(3).mean()
            full.loc[row_idx, 'y_rollmean_12'] = past.tail(12).mean()

# Build the forecast dataframe
forecast_lgb = pd.DataFrame([
    {'unique_id': uid,
     'ds': d,
     'LightGBM': v}
     for (uid, d), v in preds.items()
]
)

forecast_lgb = forecast_lgb.sort_values(['unique_id', 'ds']).reset_index(drop = True)

print('Forecast Shape: ',forecast_lgb.shape)
print(forecast_lgb.head())
    

Forecast Shape:  (2736, 3)
  unique_id         ds      LightGBM
0     TOTAL 2025-03-01  5.545778e+06
1     TOTAL 2025-04-01  4.880692e+06
2     TOTAL 2025-05-01  5.571352e+06
3     TOTAL 2025-06-01  5.372303e+06
4     TOTAL 2025-07-01  5.284816e+06


In [17]:
# Applying MinTrace Reconciliation to LightGBM forecasts

# In-sample fitted values: predict on training rows
train_pred = model.predict(train_ml[features])
fitted_lgb = train_ml[['unique_id','ds','y']].copy()
fitted_lgb['LightGBM'] = train_pred

print('Fitted Shape: ',fitted_lgb.shape)
print(fitted_lgb.head())

Fitted Shape:  (21205, 4)
  unique_id         ds          y      LightGBM
0     TOTAL 2016-04-01  2864356.0  2.846466e+06
1     TOTAL 2016-05-01  3046686.0  2.914541e+06
2     TOTAL 2016-06-01  2819655.0  2.897991e+06
3     TOTAL 2016-07-01  3011007.0  2.910875e+06
4     TOTAL 2016-08-01  2940545.0  2.910297e+06


In [19]:
# Rebuilding the hierarchy structure
df = pd.read_csv('../data/processed/hierarchy_ready.csv')
df['ds'] = pd.to_datetime(df['ds'])

to_drop = {
    'TOTAL/MUMBAI/MUMBAI-RAJKOT',
    'TOTAL/DELHI/DELHI-RAJKOT',
    'TOTAL/DEHRA DUN/DEHRA DUN-DELHI',
    'TOTAL/DEHRA DUN'
}

df = df[~df['unique_id'].isin(to_drop)].copy()

# Route-level extract with origin and route-label columns
routes_only = df[df['unique_id'].str.count('/') == 2].copy()
routes_only['origin'] = routes_only['unique_id'].str.split('/').str[1]
routes_only['route_label'] = routes_only['unique_id'].str.split('/').str[2]

# Prep for aggregate
routes_for_agg = routes_only.drop(columns = ['unique_id']).copy()
routes_for_agg['total'] = 'TOTAL'

spec = [
    ['total'],
    ['total','origin'],
    ['total','origin','route_label']
]

from hierarchicalforecast.utils import aggregate
Y_df, S_df, tags = aggregate(routes_for_agg,spec)

print("S_df: ",S_df.shape)
print("Tags: ",{k: len(v) for k, v in tags.items()})


S_df:  (228, 198)
Tags:  {'total': 1, 'total/origin': 30, 'total/origin/route_label': 197}


In [21]:
# Reconcile LightGBM
reconcilers = [
    MinTrace(method = 'ols'),
    MinTrace(method = 'mint_shrink')
]

hrec = HierarchicalReconciliation(reconcilers = reconcilers)

print("Y_df sample IDs: ", Y_df['unique_id'].unique()[:3])

print("forecast_lgb sample IDs: ", forecast_lgb['unique_id'].unique()[:3])

print("Match count: ", set(forecast_lgb['unique_id']).issubset(set(Y_df['unique_id'])))

Y_df sample IDs:  ['TOTAL' 'TOTAL/AGARTALA' 'TOTAL/AHMEDABAD']
forecast_lgb sample IDs:  ['TOTAL' 'TOTAL/AGARTALA' 'TOTAL/AGARTALA/AGARTALA-GUWAHATI']
Match count:  True


In [22]:
# Fit values aligned to Y_df identifers
fitted_lgb['ds'] = pd.to_datetime(fitted_lgb['ds'])

# Train-test split on Y_df
split_date = pd.Timestamp('2025-03-01')
Y_train_lgb = Y_df[Y_df['ds'] < split_date].copy()

# Merge LightGBM fitted values into Y_train_lgb
Y_train_lgb = Y_train_lgb.merge(fitted_lgb[['unique_id', 'ds', 'LightGBM']], on = ['unique_id', 'ds'], how = 'left')

# Missing values in fitted_lgb
print('Y_train_lgb shape: ',Y_train_lgb.shape)
print('Missing LightGBM fitted vals: ',Y_train_lgb['LightGBM'].isna().sum())

# The first 12 months per series were dropped during feature engineering for lag features. Filling those NaN with actuals as a reasonable proxy
Y_train_lgb['LightGBM'] = Y_train_lgb['LightGBM'].fillna(Y_train_lgb['y'])

print("After handling. missing values: ",Y_train_lgb['LightGBM'].isna().sum())

Y_train_lgb shape:  (23941, 4)
Missing LightGBM fitted vals:  2736
After handling. missing values:  0


In [23]:
# Build the forecast input - base LightGBM forecasts
Y_hat_lgb = forecast_lgb[['unique_id','ds','LightGBM']].copy()
Y_hat_lgb['ds'] = pd.to_datetime(Y_hat_lgb['ds'])

# Apply reconciliation
Y_rec_lgb = hrec.reconcile(
    Y_hat_df = Y_hat_lgb,
    Y_df = Y_train_lgb,
    S_df = S_df,
    tags = tags
)

print('Reconciled LightGBM forecasts shape: ',Y_rec_lgb.shape)
print('Columns: ',Y_rec_lgb.columns.to_list())
print(Y_rec_lgb.head())

Reconciled LightGBM forecasts shape:  (2736, 5)
Columns:  ['unique_id', 'ds', 'LightGBM', 'LightGBM/MinTrace_method-ols', 'LightGBM/MinTrace_method-mint_shrink']
  unique_id         ds      LightGBM  LightGBM/MinTrace_method-ols  \
0     TOTAL 2025-03-01  5.545778e+06                  5.559196e+06   
1     TOTAL 2025-04-01  4.880692e+06                  4.915733e+06   
2     TOTAL 2025-05-01  5.571352e+06                  5.578444e+06   
3     TOTAL 2025-06-01  5.372303e+06                  5.379055e+06   
4     TOTAL 2025-07-01  5.284816e+06                  5.298866e+06   

   LightGBM/MinTrace_method-mint_shrink  
0                          5.947221e+06  
1                          5.738603e+06  
2                          5.681748e+06  
3                          5.555284e+06  
4                          5.521016e+06  


In [28]:
# Compute MASE/RMSE for all LightGBM variants
def rmse (y_true, y_pred):
    return (np.sqrt(np.mean((y_true - y_pred) ** 2)))

def mase (y_true, y_pred, y_train, season_length = 12):
    naive_errors = np.abs(y_train[season_length:] - y_train[:-season_length])
    scale = naive_errors.mean()
    if scale == 0:
        return np.nan
    return np.mean(np.abs(y_true - y_pred)) / scale

# Test set actuals
Y_test_lgb = Y_df[Y_df['ds'] >= split_date].copy()

# Merge reconciled forecasts with actuals
eval_lgb = Y_test_lgb.merge(Y_rec_lgb, on = ['unique_id','ds'], how = 'left')

# Training history per series for MASE scaling
train_history = Y_train_lgb.groupby('unique_id')['y'].apply(lambda s: s.values).to_dict()

forecast_cols = ['LightGBM','LightGBM/MinTrace_method-ols','LightGBM/MinTrace_method-mint_shrink']

records = []

for uid, g in eval_lgb.groupby('unique_id'):
    y_true = g['y'].values
    y_train_s = train_history[uid]
    for col in forecast_cols:
        y_pred = g[col].values
        records.append({
            'unique_id':uid,
            'forecast': col,
            'rmse': rmse(y_true, y_pred),
            'mase': mase(y_true, y_pred, y_train_s, season_length = 12)
        })

metrics_lgb = pd.DataFrame(records)

metrics_lgb['level'] = metrics_lgb['unique_id'].apply(
    lambda x: 'TOTAL' if '/' not in x
    else ('Airport' if x.count('/') == 1 else 'Route')
)

# Aggregate by level and forecast
lgb_summary = metrics_lgb.groupby(['forecast','level'])['mase'].mean().round(3).reset_index()
lgb_pivot = lgb_summary.pivot(index = 'forecast', columns = 'level', values = 'mase')[['TOTAL', 'Airport', 'Route']]

print("LightGBM MASE by hierarchy level: ")
print(lgb_pivot.to_string())

LightGBM MASE by hierarchy level: 
level                                 TOTAL  Airport  Route
forecast                                                   
LightGBM                              0.260    0.720  0.749
LightGBM/MinTrace_method-mint_shrink  0.198    0.533  0.755
LightGBM/MinTrace_method-ols          0.257    1.145  1.204


In [29]:
# Load the base forecast and reconciliation summary
reconciled_summary = pd.read_csv('../outputs/reconciled_summary.csv', header = [0,1], index_col = 0)

print("Reconciled Summary: ")
print(reconciled_summary.to_string())

Reconciled Summary: 
base     base           ols                mint_shrink               
TOTAL Airport  Route  TOTAL Airport  Route       TOTAL Airport  Route
0.230   0.618  0.755  0.230   0.713  0.837       0.205   0.594  0.801
0.241   0.599  0.756  0.241   0.599  0.808       0.332   0.594  0.842
0.241   0.603  0.799  0.241   0.603  0.799       0.241   0.603  0.799
0.287   0.644  0.855  0.287   0.644  0.855       0.287   0.644  0.855


In [30]:
print(reconciled_summary.index.to_list())
print(reconciled_summary.shape)

[0.23, 0.241, 0.241, 0.287]
(4, 8)


In [31]:
reconciled_metrics = pd.read_csv('../outputs/reconciled_metrics.csv')

reconciled_metrics['level'] = reconciled_metrics['unique_id'].str.count('/').map({0:'TOTAL',1:'Airport',2:'Route'})

all_metrics = pd.concat([reconciled_metrics,metrics_lgb], ignore_index= True)

unified = all_metrics.groupby(['forecast','level'])['mase'].mean().round(3)

unified_pivot = unified.unstack()[['TOTAL','Airport','Route']]

print(unified_pivot.to_string())

level                                      TOTAL  Airport  Route
forecast                                                        
AutoARIMA                                  0.230    0.618  0.755
AutoARIMA/MinTrace_method-mint_shrink      0.205    0.594  0.801
AutoARIMA/MinTrace_method-ols              0.230    0.713  0.837
AutoETS                                    0.241    0.599  0.756
AutoETS/MinTrace_method-mint_shrink        0.332    0.594  0.842
AutoETS/MinTrace_method-ols                0.241    0.599  0.808
LightGBM                                   0.260    0.720  0.749
LightGBM/MinTrace_method-mint_shrink       0.198    0.533  0.755
LightGBM/MinTrace_method-ols               0.257    1.145  1.204
Naive                                      0.241    0.603  0.799
Naive/MinTrace_method-mint_shrink          0.241    0.603  0.799
Naive/MinTrace_method-ols                  0.241    0.603  0.799
SeasonalNaive                              0.287    0.644  0.855
SeasonalNaive/MinTrace_me

#### LightGBM with MinTrace shrink reconciliation tops the leaderboard at TOTAL and Airport levels with MASE of 0.198 and 0.533 respectively. At the Route level, LightGBM base forecast outperforms with a MASE of 0.749. At the Airport and Route levels, LightGBM with ols reconciliation was disastrous with MASE of 1.145 and 1.204 respectively. Overall, the ML model beats per-series statistical models at every level when paired with the correct reconciliation technique.